#1차 BO

In [10]:
!pip install scikit-optimize

In [11]:
from skopt import Optimizer
from skopt.space import Real

# 변수 범위 정의
space = [
    Real(0.1, 0.5, name='Esculetin_mM'),
    Real(1.0, 5.0, name='Reaction_OD600'),
    Real(20.0, 28.0, name='Reaction_Time_h')
]

# Gaussian Process 기반 BO Optimizer 설정
optimizer = Optimizer(
    dimensions=space,
    base_estimator="GP",
    acq_func="EI",
    n_initial_points=6,
    random_state=42
)
#-----------------------------------------------------------------------------
# 초기 6개 실험 조건 및 HPLC 결과

initial_X = [
    [0.27, 4.3, 26.5],
    [0.45, 2.8, 24.0],
    [0.33, 3.0, 25.8],
    [0.16, 2.0, 20.3],
    [0.12, 3.9, 22.9],
    [0.40, 1.2, 27.6]
]

# 실제 HPLC값 대입 : 값에는 -를 붙여서 써야하므로 initial_Y설정
hplc_results = [59.73035, 89.929575, 71.5388, 36.291, 25.32945, 10.59065]
initial_Y = [-peak_area for peak_area in hplc_results]

# BO 모델 학습
optimizer.tell(initial_X, initial_Y)

# 차기 추천 후보조건 2개 산출
next_candidates = optimizer.ask(n_points=2)
print("BO 추천 다음 검증 조건:", next_candidates)

BO 추천 다음 검증 조건: [[0.10189464825824729, 2.6587257298090474, 27.637778098686674], [0.4288626853088968, 4.984817637670069, 26.650873485498572]]


### 1차 BO 후보조건의 GP 예측값 및 EI 계산

In [12]:
import numpy as np
import pandas as pd
from scipy.stats import norm


# 1. 1차 BO에서 추천된 후보조건
candidate_X = np.asarray(
    next_candidates,
    dtype=float
)


# 2. 초기 6개 관측값으로 학습한 GP 모델 가져오기
gp_model = optimizer.models[-1]


# 3. GP 입력 형식으로 후보조건 변환
X_transformed = optimizer.space.transform(
    candidate_X.tolist()
)


# 4. GP 예측 평균 및 예측 표준편차
pred_y_neg, pred_sd = gp_model.predict(
    X_transformed,
    return_std=True
)


# 5. 목적함수가 -peak area이므로
#    다시 양의 peak area로 변환
pred_peak_area = -np.asarray(
    pred_y_neg,
    dtype=float
)


# 6. 근사 95% GP 예측구간
lower_95 = (
    pred_peak_area
    - 1.96 * pred_sd
)
upper_95 = (
    pred_peak_area
    + 1.96 * pred_sd
)


# 7. 현재까지 관측된 최고 peak area
#    (초기 6개 실험 결과)
best_observed_area = max(
    hplc_results
)
# 목적함수는 -peak area이므로
# 가장 좋은 관측값 = Y의 최솟값
best_y = min(
    optimizer.yi
)


# 8. Expected Improvement (EI) 계산
xi = 0.01
if optimizer.acq_func_kwargs is not None:
    xi = optimizer.acq_func_kwargs.get(
        "xi",
        0.01
    )
improvement = (
    best_y
    - pred_y_neg
    - xi
)
z = np.divide(
    improvement,
    pred_sd,
    out=np.zeros_like(improvement),
    where=pred_sd > 0
)
ei = (
    improvement * norm.cdf(z)
    + pred_sd * norm.pdf(z)
)
ei = np.maximum(
    ei,
    0
)


# 9. 결과표 작성
prediction_table = pd.DataFrame({
    "Candidate": ["BO1_C1", "BO1_C2"],
    "Esculetin_mM": candidate_X[:, 0],
    "Reaction_OD600": candidate_X[:, 1],
    "Reaction_Time_h": candidate_X[:, 2],
    "Predicted_peak_area": pred_peak_area,
    "GP_predictive_SD": pred_sd,
    "Approx_95%_lower": lower_95,
    "Approx_95%_upper": upper_95,
    "EI": ei
})


print("\n1차 BO 후보조건의 GP 예측 결과:")
print(
    prediction_table
    .round(2)
    .to_string(index=False)
)


1차 BO 후보조건의 GP 예측 결과:
Candidate  Esculetin_mM  Reaction_OD600  Reaction_Time_h  Predicted_peak_area  GP_predictive_SD  Approx_95%_lower  Approx_95%_upper   EI
   BO1_C1          0.10            2.66            27.64                83.19             14.26             55.24            111.13 2.94
   BO1_C2          0.43            4.98            26.65                49.86             26.58             -2.24            101.96 0.77


#2차 BO

In [13]:
#기준조건+추천받은 조건
X_round2 = [
[0.10, 2.7, 27.6],
 [0.43, 5.0, 26.7],
  [0.2, 3.0, 24.0]
             ]

# 실제 결과값
hplc_results_round2 = [18.892, 83.1772, 44.0013]
Y_round2 = [-peak_area for peak_area in hplc_results_round2]

# 기존 optimizer에 2차 결과 누적 학습
optimizer.tell(X_round2, Y_round2)

# 3차 실험조건 추천
next_candidates_r3 = optimizer.ask(n_points=1)
print("3차 실험 후보조건:", next_candidates_r3)

3차 실험 후보조건: [[0.5, 3.639063831912961, 20.0]]


### 2차 BO 후보조건의 GP 예측값 및 EI 계산

In [14]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# 1. 2차 BO에서 추천된 후보조건
candidate_X_r3 = np.asarray(
    next_candidates_r3,
    dtype=float
)


# 2. 현재까지 학습된 GP 모델 가져오기
#    (초기 6개 + 2차 BO 결과가 반영된 모델)
gp_model_r3 = optimizer.models[-1]


# 3. GP 입력 형식으로 후보조건 변환
X_transformed_r3 = optimizer.space.transform(
    candidate_X_r3.tolist()
)


# 4. GP 예측 평균 및 예측 표준편차
pred_y_neg_r3, pred_sd_r3 = gp_model_r3.predict(
    X_transformed_r3,
    return_std=True
)


# 5. 목적함수가 -peak area이므로
#    다시 양의 peak area로 변환
pred_peak_area_r3 = -np.asarray(
    pred_y_neg_r3,
    dtype=float
)


# 6. 근사 95% GP 예측구간
lower_95_r3 = (
    pred_peak_area_r3
    - 1.96 * pred_sd_r3
)

upper_95_r3 = (
    pred_peak_area_r3
    + 1.96 * pred_sd_r3
)


# 7. 현재까지 관측된 최고 peak area
#    초기 6개 + 2차 BO 결과 + 기준조건
all_peak_area_r3 = [
    59.73035,
    89.929575,
    71.5388,
    36.291,
    25.32945,
    10.59065,
    18.892,
    83.1772,
    44.0013
]

best_observed_area_r3 = max(
    all_peak_area_r3
)

# 목적함수는 -peak area이므로
# 가장 좋은 관측값 = Y의 최솟값
best_y_r3 = min(
    optimizer.yi
)


# 8. Expected Improvement (EI) 계산
xi_r3 = 0.01
if optimizer.acq_func_kwargs is not None:
    xi_r3 = optimizer.acq_func_kwargs.get(
        "xi",
        0.01
    )
improvement_r3 = (
    best_y_r3
    - pred_y_neg_r3
    - xi_r3
)
z_r3 = np.divide(
    improvement_r3,
    pred_sd_r3,
    out=np.zeros_like(improvement_r3),
    where=pred_sd_r3 > 0
)
ei_r3 = (
    improvement_r3 * norm.cdf(z_r3)
    + pred_sd_r3 * norm.pdf(z_r3)
)
ei_r3 = np.maximum(
    ei_r3,
    0
)


# 9. 결과표 작성
prediction_table_r3 = pd.DataFrame({
    "Candidate": ["BO2_C1"],
    "Esculetin_mM": candidate_X_r3[:, 0],
    "Reaction_OD600": candidate_X_r3[:, 1],
    "Reaction_Time_h": candidate_X_r3[:, 2],
    "Predicted_peak_area": pred_peak_area_r3,
    "GP_predictive_SD": pred_sd_r3,
    "Approx_95%_lower": lower_95_r3,
    "Approx_95%_upper": upper_95_r3,
    "EI": ei_r3
})


print("\n3차 실험 후보조건의 GP 예측 결과:")
print(
    prediction_table_r3
    .round(2)
    .to_string(index=False)
)


3차 실험 후보조건의 GP 예측 결과:
Candidate  Esculetin_mM  Reaction_OD600  Reaction_Time_h  Predicted_peak_area  GP_predictive_SD  Approx_95%_lower  Approx_95%_upper    EI
   BO2_C1           0.5            3.64             20.0                109.1             12.94             83.74            134.47 19.56
